# Mask2Former VerSe Semantic Multi-Seed Runner

Train/evaluate three seeds and aggregate locked metrics as mean ± std.

## 1. User Configuration

This notebook runs multiple seeds. Most edits should stay in this cell.

In [ ]:
from pathlib import Path

# Experiment identity
TASK = 'semantic'
EXPERIMENT_NAME = 'baseline'
SOURCE_TAG = 'baseline-source'
DATASET_TAG = 'verse2d-ade20k-baseline'

# Select one or more backbones. For parallel Kaggle sessions, run notebook copies with one backbone each.
BACKBONES = ['R50', 'SwinT']

# Kaggle dataset inputs. Upload source/configs/weights/data as datasets and adjust these paths only.
SOURCE_INPUT_DIR = Path('/kaggle/input/datasets/trungminh815/source-m2f')
CONFIG_ROOT = Path('/kaggle/input/datasets/trungminh815/m2f-configs-and-weights/configs')
WEIGHTS_ROOT = Path('/kaggle/input/datasets/trungminh815/m2f-configs-and-weights/weights')
VERSE_ROOT = Path('/kaggle/input/datasets/trungminh815/verse2d/ade20k')
OFFLINE_DEPENDENCIES_DIR = Path('/kaggle/input/datasets/trungminh815/m2f-offline-dependencies')

# Working/output locations
WORKING_REPO = Path('/kaggle/working/Mask2Former')
OUTPUT_BASE_DIR = Path('/kaggle/working/experiments')
RESET_WORKING_SOURCE = True

# Launch behavior
RUN_TRAINING = True
RUN_EVALUATION = True
RESUME = False
NUM_GPUS = 1
NUM_MACHINES = 1
MACHINE_RANK = 0
DIST_URL = 'tcp://127.0.0.1:64152'
STOP_ON_BACKBONE_FAILURE = True
# Save-Version mode keeps the notebook kernel alive long enough to write logs/summaries.
# If a subprocess fails, the row is marked failed instead of aborting papermill.
SAVE_VERSION_SAFE_MODE = True
FAIL_NOTEBOOK_ON_ERROR = False
LOG_HEARTBEAT_SECONDS = 60
SEED = 42
SEEDS = [42, 3407, 2026]

# Use val during training checkpoints; use test for final report metrics.
TRAIN_EVAL_SPLIT = 'val'
EVAL_SPLIT = 'test'

# Semantic task settings
TASK_DATASET_PREFIX = 'verse_semantic'
TRAIN_DATASET_NAME = 'verse_semantic_train'
NUM_CLASSES = 4
CLASS_NAMES = ['background', 'cervical', 'thoracic', 'lumbar']

# Common config overrides. Set values to None to keep the base config value.
DEFAULT_EXTRA_OPTS = {
    'SOLVER.IMS_PER_BATCH': '4',
    'SOLVER.BASE_LR': None,
    'SOLVER.MAX_ITER': None,
    'SOLVER.CHECKPOINT_PERIOD': None,
    'TEST.EVAL_PERIOD': None,
    'DATALOADER.NUM_WORKERS': '2',
}

# Extra Detectron2 opts applied to every backbone. Good for Priority 1/2 experiments.
COMMON_EXTRA_OPTS = []

# Optional backbone-specific overrides.
BACKBONE_EXTRA_OPTS = {
    # 'SwinT': ['SOLVER.IMS_PER_BATCH', '1'],
}

# For evaluation-only sweeps, set RUN_TRAINING=False and point each backbone to a trained checkpoint.
CHECKPOINTS_BY_BACKBONE = {
    # 'R50': '/kaggle/input/my-checkpoints/R50/model_final.pth',
}

BACKBONE_REGISTRY = {
    'R50': {
        'config': 'verse/verse_ade20k_semantic_R50.yaml',
        'weights': 'ade20k_semantic_R50.pkl',
    },
    'SwinT': {
        'config': 'verse/verse_ade20k_semantic_swin_t.yaml',
        'weights': 'ade20k_semantic_swin_tiny.pkl',
    },
}

## 2. Environment Setup

Run these cells on Kaggle before training. They keep the dependency setup from the working Mask2Former notebook and repair NumPy when Kaggle starts with an incompatible NumPy 2.x stack.

In [ ]:
import os
import sys
import json
import csv
import shutil
import subprocess
from pathlib import Path
from importlib import metadata


def run_pip(args):
    subprocess.run([sys.executable, '-m', 'pip'] + list(args), check=True)


def installed_version(package_name):
    try:
        return metadata.version(package_name)
    except metadata.PackageNotFoundError:
        return None


def fresh_python_numpy_version():
    code = 'import numpy as np; print(np.__version__)'
    try:
        return subprocess.check_output([sys.executable, '-c', code], text=True).strip()
    except Exception as exc:
        return f'ERROR: {exc}'


def _unique_paths(paths):
    seen = set()
    unique = []
    for path in paths:
        path = Path(path)
        key = str(path)
        if key not in seen:
            unique.append(path)
            seen.add(key)
    return unique


def dependency_roots():
    root = Path(OFFLINE_DEPENDENCIES_DIR)
    has_packages = root.exists() and any(
        root.rglob(pattern)
        for pattern in ('*.whl', '*.tar.gz', '*.zip')
    )
    print('Offline dependency root:', root, 'exists=' + str(root.exists()), 'packages=' + str(has_packages))
    if not has_packages:
        if Path('/kaggle/input').exists():
            print('Available /kaggle/input entries:')
            for child in sorted(Path('/kaggle/input').iterdir()):
                print(' -', child)
        raise FileNotFoundError(
            'Could not find offline wheels at OFFLINE_DEPENDENCIES_DIR. '
            'Attach dataset trungminh815/m2f-offline-dependencies and use: '
            '/kaggle/input/datasets/trungminh815/m2f-offline-dependencies'
        )
    return [root]


def offline_package_files():
    package_files = []
    for root in dependency_roots():
        for pattern in ('*.whl', '*.tar.gz', '*.zip'):
            package_files.extend(root.rglob(pattern))
    return sorted(_unique_paths(package_files))


def offline_find_links_args():
    package_files = offline_package_files()
    link_dirs = _unique_paths([p.parent for p in package_files])
    print(f'Offline package files found: {len(package_files)}')
    print('Offline package sample:')
    for package in package_files[:20]:
        print(' -', package.name)
    if len(package_files) > 20:
        print(' - ...')
    print(f'Offline --find-links directories: {len(link_dirs)}')
    for link_dir in link_dirs[:20]:
        print(' -', link_dir)
    if len(link_dirs) > 20:
        print(' - ...')

    args = []
    for link_dir in link_dirs:
        args.extend(['--find-links', str(link_dir)])
    return args


def require_offline_file(substring):
    matches = [p for p in offline_package_files() if substring in p.name]
    if not matches:
        raise FileNotFoundError(f'Missing required offline package containing: {substring}')
    print('Found required package:', matches[0].name)


def run_pip_offline_install(packages, no_deps=False):
    args = ['install', '--no-index', '--no-cache-dir']
    if no_deps:
        args.append('--no-deps')
    run_pip(args + offline_find_links_args() + list(packages))


def setup_numpy_environment():
    # Kaggle Save Version cannot survive intentional kernel restarts. The current
    # notebook kernel may already have NumPy 2.x loaded before this cell runs, so
    # the important check is the fresh subprocess used for train.py/eval.
    current_numpy = installed_version('numpy')
    print('Installed NumPy version before setup:', current_numpy)

    if current_numpy is None or not current_numpy.startswith('1.'):
        # Validate the exact Python 3.12 NumPy wheel before uninstalling anything.
        require_offline_file('numpy-1.26.4-cp312')
        print('Installing NumPy 1.26.4 for Detectron2/Mask2Former subprocesses...')
        run_pip(['uninstall', '-y', 'numpy', 'detectron2'])
        run_pip_offline_install(['numpy==1.26.4'])

    fresh_numpy = fresh_python_numpy_version()
    print('Fresh Python NumPy version:', fresh_numpy)
    if not fresh_numpy.startswith('1.'):
        raise RuntimeError(f'Fresh Python still does not see NumPy 1.x: {fresh_numpy}')

    global np
    import numpy as np
    print('Notebook-kernel NumPy version:', np.__version__)
    if not np.__version__.startswith('1.'):
        print('WARNING: current notebook kernel still has NumPy 2.x loaded. This is expected in Kaggle Save Version; train/eval subprocesses use fresh Python with NumPy 1.x.')


setup_numpy_environment()


In [ ]:
# Install all dependencies from the offline wheels/source dataset.
# Keep NumPy 1.26.4 because Detectron2/pycocotools are unstable with Kaggle's
# default NumPy 2.x in this project. Kaggle may expand .tar.gz source packages
# when mounting datasets, so source packages are installed from either package
# files or expanded setup.py directories.
run_pip(['uninstall', '-y', 'pycocotools', 'detectron2', 'numpy', 'fvcore', 'iopath', 'panopticapi'])


def _normalized_package_name(path):
    name = path.name.lower().replace('-', '_')
    for suffix in ('.tar.gz', '.whl', '.zip'):
        if name.endswith(suffix):
            name = name[:-len(suffix)]
    return name


def _offline_package_candidates(package_prefix):
    prefix = package_prefix.lower().replace('-', '_')
    return [p for p in offline_package_files() if _normalized_package_name(p).startswith(prefix)]


def _source_setup_candidates(markers):
    markers = [m.lower() for m in markers]
    matches = []
    for root in dependency_roots():
        for setup_py in root.rglob('setup.py'):
            text = str(setup_py.parent).lower()
            if any(marker in text for marker in markers):
                matches.append(setup_py.parent)
    return sorted(_unique_paths(matches))


def _add_source_to_pythonpath(source_dir):
    source_dir = Path(source_dir)
    source_text = str(source_dir)
    if source_text not in sys.path:
        sys.path.insert(0, source_text)
    current = os.environ.get('PYTHONPATH', '')
    parts = [p for p in current.split(os.pathsep) if p]
    if source_text not in parts:
        os.environ['PYTHONPATH'] = source_text + (os.pathsep + current if current else '')


def install_offline_package_or_source(package_name, source_markers=None, source_mode='pip'):
    candidates = _offline_package_candidates(package_name)
    if candidates:
        print(f'Installing {package_name} from offline package file:', candidates[0].name)
        run_pip_offline_install([package_name])
        return

    source_candidates = _source_setup_candidates(source_markers or [package_name])
    if source_candidates:
        if source_mode == 'path':
            print(f'Using {package_name} from expanded source via PYTHONPATH:', source_candidates[0])
            _add_source_to_pythonpath(source_candidates[0])
        else:
            print(f'Installing {package_name} from expanded source:', source_candidates[0])
            run_pip(['install', '--no-index', '--no-cache-dir', '--no-build-isolation', '--no-deps', str(source_candidates[0])])
        return

    raise FileNotFoundError(f'Could not find offline package file or expanded source directory for {package_name}')


# Install NumPy first, then install the rest against that ABI.
run_pip_offline_install(['numpy==1.26.4'])
run_pip_offline_install([
    'matplotlib==3.8.4',
    'Pillow==10.3.0',
    'scipy',
    'nibabel',
    'cython',
    'shapely',
    'timm',
    'h5py',
    'submitit',
    'scikit-image',
    'ninja',
    'yacs',
    'tabulate',
    'termcolor',
    'cloudpickle',
    'portalocker',
], no_deps=True)

install_offline_package_or_source('fvcore', ['fvcore'], source_mode='path')
install_offline_package_or_source('iopath', ['iopath'], source_mode='path')
run_pip_offline_install(['pycocotools'])
install_offline_package_or_source('panopticapi', ['panopticapi'], source_mode='path')
detectron2_wheels = [p for p in offline_package_files() if p.name.startswith('detectron2-') and p.suffix == '.whl']
if not detectron2_wheels:
    raise FileNotFoundError('Missing compiled Detectron2 wheel. Expected detectron2-0.6-cp312-cp312-linux_x86_64.whl in the offline dependency dataset.')
print('Installing Detectron2 wheel:', detectron2_wheels[0].name)
run_pip(['install', '--no-index', '--no-cache-dir', '--no-deps', str(detectron2_wheels[0])])

opencv_wheels = [
    p for p in offline_package_files()
    if p.name.startswith('opencv_python_headless-4.10.0.84')
]
if opencv_wheels:
    print('Installing offline OpenCV wheel:', opencv_wheels[0].name)
    run_pip(['uninstall', '-y', 'opencv-python', 'opencv-python-headless', 'opencv-contrib-python'])
    run_pip_offline_install(['opencv-python-headless==4.10.0.84'])
else:
    print('No offline opencv-python-headless==4.10.0.84 wheel found; keeping Kaggle built-in OpenCV if available.')

subprocess.run([
    sys.executable, '-c',
    'import numpy as np; import cv2; import detectron2; from detectron2 import _C; from pycocotools import mask; import fvcore; import iopath; print("NumPy", np.__version__, "OpenCV", cv2.__version__, "detectron2 _C OK")'
], check=True)


## 3. Source Setup and CUDA Operators

The notebook copies Mask2Former source from a Kaggle dataset into `/kaggle/working`. For Priority 3 experiments, upload a new source dataset version and change only `SOURCE_INPUT_DIR` in the user configuration cell.

In [ ]:
import glob
import importlib
import zipfile

TRAIN_SCRIPT_TEXT = '# Copyright (c) Facebook, Inc. and its affiliates. All Rights Reserved\n"""\nMaskFormer Training Script.\n\nThis script is a simplified version of the training script in detectron2/tools.\n"""\ntry:\n    # ignore ShapelyDeprecationWarning from fvcore\n    from shapely.errors import ShapelyDeprecationWarning\n    import warnings\n    warnings.filterwarnings(\'ignore\', category=ShapelyDeprecationWarning)\n    warnings.filterwarnings(\'ignore\', category=FutureWarning)\n    warnings.filterwarnings(\'ignore\', module=\'timm\')\nexcept:\n    pass\n\nimport copy\nimport itertools\nimport logging\nimport os\n\nfrom collections import OrderedDict\nfrom typing import Any, Dict, List, Set\n\nimport torch\n\nfrom PIL import Image\n# Pillow >=10 removed some legacy aliases still referenced by older Detectron2 code.\nif not hasattr(Image, "LINEAR"):\n    Image.LINEAR = Image.BILINEAR\nif not hasattr(Image, "CUBIC"):\n    Image.CUBIC = Image.BICUBIC\nif not hasattr(Image, "ANTIALIAS"):\n    Image.ANTIALIAS = Image.LANCZOS\n\nimport detectron2.utils.comm as comm\nfrom detectron2.checkpoint import DetectionCheckpointer\nfrom detectron2.config import get_cfg\nfrom detectron2.data import MetadataCatalog, build_detection_train_loader\nfrom detectron2.engine import (\n    DefaultTrainer,\n    default_argument_parser,\n    default_setup,\n    launch,\n)\nfrom detectron2.evaluation import (\n    CityscapesInstanceEvaluator,\n    CityscapesSemSegEvaluator,\n    COCOEvaluator,\n    COCOPanopticEvaluator,\n    DatasetEvaluators,\n    LVISEvaluator,\n    SemSegEvaluator,\n    verify_results,\n)\nfrom detectron2.projects.deeplab import add_deeplab_config, build_lr_scheduler\nfrom detectron2.solver.build import maybe_add_gradient_clipping\nfrom detectron2.utils.logger import setup_logger\n\n# MaskFormer\nfrom mask2former import (\n    InstanceSegEvaluator,\n    MaskFormerInstanceDatasetMapper,\n    MaskFormerPanopticDatasetMapper,\n    MaskFormerSemanticDatasetMapper,\n    SemanticSegmentorWithTTA,\n    add_maskformer2_config,\n)\n\n# Config-driven dataset registration\nfrom mask2former.data.datasets.register_dataset import register_all_verse_datasets\n\n\nclass Trainer(DefaultTrainer):\n    """\n    Extension of the Trainer class adapted to MaskFormer.\n    """\n\n    @classmethod\n    def build_evaluator(cls, cfg, dataset_name, output_folder=None):\n        """\n        Refactored: Metadata-driven evaluator selection.\n        """\n        if output_folder is None:\n            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")\n\n        evaluator_list = []\n        evaluator_type = MetadataCatalog.get(dataset_name).evaluator_type\n\n        # 1. Semantic Segmentation\n        if evaluator_type in ["sem_seg", "ade20k_sem_seg", "cityscapes_sem_seg"]:\n            evaluator_list.append(SemSegEvaluator(dataset_name, distributed=True, output_dir=output_folder))\n\n        # 2. Instance Segmentation\n        if evaluator_type in ["coco", "cityscapes_instance", "ade20k_instance"]:\n            evaluator_list.append(COCOEvaluator(dataset_name, output_dir=output_folder))\n\n        # 3. Panoptic Segmentation\n        if evaluator_type in ["coco_panoptic_seg", "ade20k_panoptic_seg", "cityscapes_panoptic_seg", "mapillary_vistas_panoptic_seg"]:\n            evaluator_list.append(COCOPanopticEvaluator(dataset_name, output_folder))\n\n        if len(evaluator_list) == 0:\n            raise NotImplementedError(f"No Evaluator for dataset {dataset_name} (type {evaluator_type})")\n        elif len(evaluator_list) == 1:\n            return evaluator_list[0]\n        \n        return DatasetEvaluators(evaluator_list)\n\n    @classmethod\n    def build_train_loader(cls, cfg):\n        # Universal Mapper Selection\n        mapper_name = cfg.INPUT.DATASET_MAPPER_NAME\n        mapper_map = {\n            "mask_former_semantic": MaskFormerSemanticDatasetMapper,\n            "mask_former_panoptic": MaskFormerPanopticDatasetMapper,\n            "mask_former_instance": MaskFormerInstanceDatasetMapper,\n            # Handle COCO-style LSJ mapper names that might be in configs\n            "coco_instance_lsj": MaskFormerInstanceDatasetMapper,\n            "coco_panoptic_lsj": MaskFormerPanopticDatasetMapper,\n        }\n        \n        mapper_cls = mapper_map.get(mapper_name, None)\n        mapper = mapper_cls(cfg, True) if mapper_cls else None\n        return build_detection_train_loader(cfg, mapper=mapper)\n\n    @classmethod\n    def build_lr_scheduler(cls, cfg, optimizer):\n        """\n        It now calls :func:`detectron2.solver.build_lr_scheduler`.\n        Overwrite it if you\'d like a different scheduler.\n        """\n        return build_lr_scheduler(cfg, optimizer)\n\n    @classmethod\n    def build_optimizer(cls, cfg, model):\n        weight_decay_norm = cfg.SOLVER.WEIGHT_DECAY_NORM\n        weight_decay_embed = cfg.SOLVER.WEIGHT_DECAY_EMBED\n\n        defaults = {}\n        defaults["lr"] = cfg.SOLVER.BASE_LR\n        defaults["weight_decay"] = cfg.SOLVER.WEIGHT_DECAY\n\n        norm_module_types = (\n            torch.nn.BatchNorm1d,\n            torch.nn.BatchNorm2d,\n            torch.nn.BatchNorm3d,\n            torch.nn.SyncBatchNorm,\n            # NaiveSyncBatchNorm inherits from BatchNorm2d\n            torch.nn.GroupNorm,\n            torch.nn.InstanceNorm1d,\n            torch.nn.InstanceNorm2d,\n            torch.nn.InstanceNorm3d,\n            torch.nn.LayerNorm,\n            torch.nn.LocalResponseNorm,\n        )\n\n        params: List[Dict[str, Any]] = []\n        memo: Set[torch.nn.parameter.Parameter] = set()\n        for module_name, module in model.named_modules():\n            for module_param_name, value in module.named_parameters(recurse=False):\n                if not value.requires_grad:\n                    continue\n                # Avoid duplicating parameters\n                if value in memo:\n                    continue\n                memo.add(value)\n\n                hyperparams = copy.copy(defaults)\n                if "backbone" in module_name:\n                    hyperparams["lr"] = hyperparams["lr"] * cfg.SOLVER.BACKBONE_MULTIPLIER\n                if (\n                    "relative_position_bias_table" in module_param_name\n                    or "absolute_pos_embed" in module_param_name\n                ):\n                    print(module_param_name)\n                    hyperparams["weight_decay"] = 0.0\n                if isinstance(module, norm_module_types):\n                    hyperparams["weight_decay"] = weight_decay_norm\n                if isinstance(module, torch.nn.Embedding):\n                    hyperparams["weight_decay"] = weight_decay_embed\n                params.append({"params": [value], **hyperparams})\n\n        def maybe_add_full_model_gradient_clipping(optim):\n            # detectron2 doesn\'t have full model gradient clipping now\n            clip_norm_val = cfg.SOLVER.CLIP_GRADIENTS.CLIP_VALUE\n            enable = (\n                cfg.SOLVER.CLIP_GRADIENTS.ENABLED\n                and cfg.SOLVER.CLIP_GRADIENTS.CLIP_TYPE == "full_model"\n                and clip_norm_val > 0.0\n            )\n\n            class FullModelGradientClippingOptimizer(optim):\n                def step(self, closure=None):\n                    all_params = itertools.chain(*[x["params"] for x in self.param_groups])\n                    torch.nn.utils.clip_grad_norm_(all_params, clip_norm_val)\n                    super().step(closure=closure)\n\n            return FullModelGradientClippingOptimizer if enable else optim\n\n        optimizer_type = cfg.SOLVER.OPTIMIZER\n        if optimizer_type == "SGD":\n            optimizer = maybe_add_full_model_gradient_clipping(torch.optim.SGD)(\n                params, cfg.SOLVER.BASE_LR, momentum=cfg.SOLVER.MOMENTUM\n            )\n        elif optimizer_type == "ADAMW":\n            optimizer = maybe_add_full_model_gradient_clipping(torch.optim.AdamW)(\n                params, cfg.SOLVER.BASE_LR\n            )\n        else:\n            raise NotImplementedError(f"no optimizer type {optimizer_type}")\n        if not cfg.SOLVER.CLIP_GRADIENTS.CLIP_TYPE == "full_model":\n            optimizer = maybe_add_gradient_clipping(cfg, optimizer)\n        return optimizer\n\n    @classmethod\n    def test_with_TTA(cls, cfg, model):\n        logger = logging.getLogger("detectron2.trainer")\n        # In the end of training, run an evaluation with TTA.\n        logger.info("Running inference with test-time augmentation ...")\n        model = SemanticSegmentorWithTTA(cfg, model)\n        evaluators = [\n            cls.build_evaluator(\n                cfg, name, output_folder=os.path.join(cfg.OUTPUT_DIR, "inference_TTA")\n            )\n            for name in cfg.DATASETS.TEST\n        ]\n        res = cls.test(cfg, model, evaluators)\n        res = OrderedDict({k + "_TTA": v for k, v in res.items()})\n        return res\n\n\ndef setup(args):\n    """\n    Create configs and perform basic setups.\n    """\n    cfg = get_cfg()\n    # for poly lr schedule\n    add_deeplab_config(cfg)\n    add_maskformer2_config(cfg)\n    cfg.merge_from_file(args.config_file)\n    cfg.merge_from_list(args.opts)\n    cfg.freeze()\n    default_setup(cfg, args)\n    \n    # Config-driven VerSe registration\n    if cfg.DATASETS.VERSE_ROOT:\n        register_all_verse_datasets(cfg.DATASETS.VERSE_ROOT)\n\n    # Setup logger for "mask_former" module\n    setup_logger(output=cfg.OUTPUT_DIR, distributed_rank=comm.get_rank(), name="mask2former")\n    return cfg\n\n\ndef main(args):\n    cfg = setup(args)\n\n    if args.eval_only:\n        model = Trainer.build_model(cfg)\n        DetectionCheckpointer(model, save_dir=cfg.OUTPUT_DIR).resume_or_load(\n            cfg.MODEL.WEIGHTS, resume=args.resume\n        )\n        res = Trainer.test(cfg, model)\n        if cfg.TEST.AUG.ENABLED:\n            res.update(Trainer.test_with_TTA(cfg, model))\n        if comm.is_main_process():\n            verify_results(cfg, res)\n        return res\n\n    trainer = Trainer(cfg)\n    trainer.resume_or_load(resume=args.resume)\n    return trainer.train()\n\n\nif __name__ == "__main__":\n    args = default_argument_parser().parse_args()\n    print("Command Line Args:", args)\n    launch(\n        main,\n        args.num_gpus,\n        num_machines=args.num_machines,\n        machine_rank=args.machine_rank,\n        dist_url=args.dist_url,\n        args=(args,),\n    )\n'


def resolve_existing_path(path_like):
    path = Path(path_like)
    return path if str(path_like) and path.exists() else None


def is_mask2former_source_dir(path):
    path = Path(path)
    return (path / 'mask2former').is_dir()


def unpack_zipped_source_root(source_path):
    source_path = Path(source_path)
    zip_files = sorted(source_path.glob('*.zip')) if source_path.exists() else []
    if not zip_files:
        return None

    unpack_dir = Path('/kaggle/working') / f'{SOURCE_TAG}_unzipped_source'
    if unpack_dir.exists():
        shutil.rmtree(unpack_dir)
    unpack_dir.mkdir(parents=True, exist_ok=True)

    for zip_path in zip_files:
        print(f'Unpacking source archive: {zip_path.name}')
        with zipfile.ZipFile(zip_path) as zf:
            target_dir = unpack_dir / zip_path.stem
            target_dir.mkdir(parents=True, exist_ok=True)
            zf.extractall(target_dir)

    for item in source_path.iterdir():
        if item.is_file() and item.suffix != '.zip':
            shutil.copy2(item, unpack_dir / item.name)
    return unpack_dir


def resolve_source_code_path(source_path):
    p = Path(source_path)
    candidates = [p]
    unpacked = unpack_zipped_source_root(p)
    if unpacked is not None:
        candidates.insert(0, unpacked)

    for candidate in candidates:
        if is_mask2former_source_dir(candidate):
            return candidate
        if is_mask2former_source_dir(candidate / 'Mask2Former'):
            return candidate / 'Mask2Former'
        if candidate.exists():
            for child in sorted(candidate.iterdir()):
                if child.is_dir() and is_mask2former_source_dir(child):
                    return child
    raise FileNotFoundError(
        'Could not find a Mask2Former source folder at SOURCE_INPUT_DIR. '
        'Attach the public Kaggle source dataset for the selected method and set SOURCE_INPUT_DIR to its /kaggle/input/datasets/... path.'
    )


def copy_source_to_working(source_path, destination_dir, reset=True):
    destination_dir = Path(destination_dir)
    if reset and destination_dir.exists():
        print(f'Removing existing working source: {destination_dir}')
        shutil.rmtree(destination_dir)
    if not destination_dir.exists():
        print(f'Copying source from {source_path} to {destination_dir}')
        shutil.copytree(source_path, destination_dir)
    else:
        print(f'Using existing working source: {destination_dir}')
    return destination_dir


def patch_legacy_aliases_in_train_entrypoint(train_py):
    train_py = Path(train_py)
    text = train_py.read_text(encoding='utf-8')
    marker = 'import torch\n'
    patch = '''import torch
import numpy as np

# NumPy >=1.24 removed aliases still referenced by older Detectron2 paths.
if not hasattr(np, "int"):
    np.int = int
if not hasattr(np, "float"):
    np.float = float
if not hasattr(np, "bool"):
    np.bool = bool

from PIL import Image

# Pillow >=10 removed legacy aliases still referenced by older Detectron2 code.
if not hasattr(Image, "LINEAR"):
    Image.LINEAR = Image.BILINEAR
if not hasattr(Image, "CUBIC"):
    Image.CUBIC = Image.BICUBIC
if not hasattr(Image, "ANTIALIAS"):
    Image.ANTIALIAS = Image.LANCZOS
'''
    needs_numpy_patch = 'np.int = int' not in text
    needs_pillow_patch = 'Image.LINEAR = Image.BILINEAR' not in text
    if not needs_numpy_patch and not needs_pillow_patch:
        return
    if marker not in text:
        raise RuntimeError(f'Cannot patch legacy aliases in {train_py}: import torch marker not found')
    # If a previous notebook version already inserted only the Pillow shim, replace that block too.
    pillow_start = text.find('from PIL import Image\n\n# Pillow >=10 removed legacy aliases')
    if pillow_start != -1 and needs_numpy_patch:
        text = text[:pillow_start] + text[text.find('import detectron2.utils.comm as comm', pillow_start):]
    train_py.write_text(text.replace(marker, patch, 1), encoding='utf-8')
    print(f'Patched legacy NumPy/Pillow aliases in {train_py}')


def ensure_train_entrypoint(repo_dir):
    repo_dir = Path(repo_dir)
    train_py = repo_dir / 'train.py'
    if train_py.exists():
        print(f'Using existing training entrypoint: {train_py}')
        patch_legacy_aliases_in_train_entrypoint(train_py)
        return train_py
    print(f'Creating training entrypoint because source dataset does not include train.py: {train_py}')
    train_py.write_text(TRAIN_SCRIPT_TEXT, encoding='utf-8')
    patch_legacy_aliases_in_train_entrypoint(train_py)
    return train_py


def detect_cuda_home():
    if os.environ.get('CUDA_HOME'):
        print(f'Using CUDA_HOME={os.environ["CUDA_HOME"]}')
        return os.environ['CUDA_HOME']
    try:
        nvcc_path = subprocess.check_output(['which', 'nvcc'], text=True).strip()
        cuda_home = str(Path(nvcc_path).parent.parent)
        os.environ['CUDA_HOME'] = cuda_home
        print(f'Found CUDA via nvcc: {cuda_home}')
        return cuda_home
    except subprocess.CalledProcessError:
        for candidate in glob.glob('/usr/local/cuda-*') + ['/usr/local/cuda']:
            if Path(candidate).exists():
                os.environ['CUDA_HOME'] = candidate
                print(f'Found CUDA via path search: {candidate}')
                return candidate
    raise RuntimeError('CUDA not found. Enable Kaggle GPU before running training.')


def run_in_dir(cwd, command):
    print('Running:', ' '.join(map(str, command)))
    subprocess.run(command, cwd=str(cwd), check=True)


def compile_mask2former_ops(repo_dir):
    detect_cuda_home()
    ops_dir = Path(repo_dir) / 'mask2former' / 'modeling' / 'pixel_decoder' / 'ops'
    if not ops_dir.exists():
        raise FileNotFoundError(f'Cannot find Mask2Former ops directory: {ops_dir}')

    print(f'Compiling custom ops in {ops_dir}')
    for pattern in ['build', '*.egg-info', 'dist']:
        for p in ops_dir.glob(pattern):
            if p.is_dir():
                shutil.rmtree(p)
            else:
                p.unlink()

    run_in_dir(ops_dir, [sys.executable, '-m', 'pip', 'install', '.'])
    make_sh = ops_dir / 'make.sh'
    if make_sh.exists():
        run_in_dir(ops_dir, ['sh', 'make.sh'])
    importlib.invalidate_caches()
    print('Custom ops compilation finished.')


source_path = resolve_source_code_path(SOURCE_INPUT_DIR)
WORKING_REPO = copy_source_to_working(source_path, WORKING_REPO, reset=RESET_WORKING_SOURCE)
ensure_train_entrypoint(WORKING_REPO)
compile_mask2former_ops(WORKING_REPO)

## 4. Multi-Seed Experiment Runner

This section trains/evaluates every selected seed and backbone, aggregates locked metrics as mean ± std, and deletes large checkpoint artifacts after evaluation.

In [ ]:
import time
from collections import defaultdict


def path_from_root(root, relative_path):
    p = Path(relative_path)
    return p if p.is_absolute() else Path(root) / p


def require_file(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'{label} not found: {path}')
    return path


def tuple_override(dataset_name):
    return f'("{dataset_name}",)'


def build_common_opts(backbone_name, spec, output_dir, split, weights_path):
    opts = [
        'DATASETS.TRAIN', tuple_override(TRAIN_DATASET_NAME),
        'DATASETS.TEST', tuple_override(f'{TASK_DATASET_PREFIX}_{split}'),
        'DATASETS.VERSE_ROOT', str(VERSE_ROOT),
        'MODEL.WEIGHTS', str(weights_path),
        'MODEL.SEM_SEG_HEAD.NUM_CLASSES', str(NUM_CLASSES),
        'OUTPUT_DIR', str(output_dir),
        'SEED', str(SEED),
    ]
    for key, value in DEFAULT_EXTRA_OPTS.items():
        if value is not None:
            opts.extend([key, str(value)])
    opts.extend(COMMON_EXTRA_OPTS)
    opts.extend(BACKBONE_EXTRA_OPTS.get(backbone_name, []))
    return [str(x) for x in opts]


def should_print_log_line(line, last_print_time, heartbeat_seconds):
    important_tokens = [
        'iter:', 'eta:', 'total_loss:', 'Evaluation results', 'copypaste:',
        'Average Precision', 'Average Recall', 'Saved metrics', 'Wrote ',
        'ERROR', 'Error', 'Traceback', 'Exception', 'failed', 'finished',
        'Command finished', 'Saving checkpoint', 'model_final', 'CUDA out of memory',
    ]
    now = time.time()
    if any(token in line for token in important_tokens):
        return True, now
    if now - last_print_time >= heartbeat_seconds:
        return True, now
    return False, last_print_time


def run_streaming_command(cmd, cwd, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env['PYTHONPATH'] = f'{cwd}:{env.get("PYTHONPATH", "")}'
    env.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
    print('\n' + '=' * 80)
    print('Running command:')
    print(' '.join(map(str, cmd)))
    print(f'Full log: {log_path}')
    print('=' * 80)
    started = time.time()
    last_print_time = started
    with open(log_path, 'w', encoding='utf-8') as log_file:
        process = subprocess.Popen(
            [str(x) for x in cmd],
            cwd=str(cwd),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            log_file.write(line)
            log_file.flush()
            if SAVE_VERSION_SAFE_MODE:
                do_print, last_print_time = should_print_log_line(line, last_print_time, LOG_HEARTBEAT_SECONDS)
                if do_print:
                    print(line, end='')
            else:
                print(line, end='')
        return_code = process.wait()
    elapsed = time.time() - started
    print(f'Command finished with exit code {return_code} after {elapsed / 60:.1f} minutes')
    if return_code != 0:
        raise RuntimeError(f'Command failed with exit code {return_code}. See {log_path}')
def locate_checkpoint(output_dir, fallback_weights=None):
    output_dir = Path(output_dir)
    final = output_dir / 'model_final.pth'
    if final.exists():
        return final
    last_checkpoint = output_dir / 'last_checkpoint'
    if last_checkpoint.exists():
        name = last_checkpoint.read_text().strip()
        candidate = output_dir / name
        if candidate.exists():
            return candidate
    candidates = sorted(output_dir.glob('*.pth'), key=lambda p: p.stat().st_mtime)
    if candidates:
        return candidates[-1]
    if fallback_weights and Path(fallback_weights).exists():
        return Path(fallback_weights)
    raise FileNotFoundError(f'No checkpoint found in {output_dir}')


def write_summary_files(rows, summary_dir):
    summary_dir = Path(summary_dir)
    summary_dir.mkdir(parents=True, exist_ok=True)
    json_path = summary_dir / 'summary.json'
    csv_path = summary_dir / 'summary.csv'
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(rows, f, indent=2)
    fieldnames = sorted({key for row in rows for key in row.keys()})
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)
    print(f'Wrote {json_path}')
    print(f'Wrote {csv_path}')


def print_summary(rows):
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(row)

In [ ]:
from PIL import Image
from pycocotools import mask as mask_util


def compute_semantic_metrics(pred_json_path, gt_root, split, num_classes, class_names):
    pred_json_path = Path(pred_json_path)
    if not pred_json_path.exists():
        raise FileNotFoundError(f'Semantic predictions not found: {pred_json_path}')

    with open(pred_json_path, 'r', encoding='utf-8') as f:
        predictions = json.load(f)

    by_file = defaultdict(list)
    for pred in predictions:
        by_file[Path(pred['file_name']).name].append(pred)

    gt_dir = Path(gt_root) / split / 'annotations_semantic'
    if not gt_dir.exists():
        raise FileNotFoundError(f'Semantic GT directory not found: {gt_dir}')

    confusion = np.zeros((num_classes, num_classes), dtype=np.int64)
    evaluated_images = 0

    for filename, entries in by_file.items():
        gt_path = gt_dir / filename
        if not gt_path.exists():
            continue
        gt = np.array(Image.open(gt_path), dtype=np.int64)
        pred_label = np.zeros(gt.shape, dtype=np.int64)
        for entry in entries:
            category_id = int(entry['category_id'])
            if category_id < 0 or category_id >= num_classes:
                continue
            mask = mask_util.decode(entry['segmentation']).astype(bool)
            if mask.shape != pred_label.shape:
                continue
            pred_label[mask] = category_id
        valid = (gt != 255) & (gt >= 0) & (gt < num_classes)
        hist = np.bincount(
            num_classes * gt[valid].reshape(-1) + pred_label[valid].reshape(-1),
            minlength=num_classes ** 2,
        ).reshape(num_classes, num_classes)
        confusion += hist
        evaluated_images += 1

    tp = np.diag(confusion).astype(float)
    gt_count = confusion.sum(axis=1).astype(float)
    pred_count = confusion.sum(axis=0).astype(float)
    union = gt_count + pred_count - tp
    iou = np.divide(tp, union, out=np.full_like(tp, np.nan), where=union > 0)
    dice = np.divide(2 * tp, gt_count + pred_count, out=np.full_like(tp, np.nan), where=(gt_count + pred_count) > 0)
    acc = np.divide(tp, gt_count, out=np.full_like(tp, np.nan), where=gt_count > 0)

    fg = list(range(1, num_classes)) if num_classes > 1 else list(range(num_classes))
    pixel_accuracy = float(np.nansum(tp) / max(confusion.sum(), 1))
    mean_accuracy = float(np.nanmean(acc))
    mean_iou = float(np.nanmean(iou))
    mean_dice = float(np.nanmean(dice))
    foreground_mean_iou = float(np.nanmean(iou[fg]))
    foreground_mean_dice = float(np.nanmean(dice[fg]))
    metrics = {
        'evaluated_images': int(evaluated_images),
        'pixel_accuracy': pixel_accuracy,
        'pixel_accuracy_percent': pixel_accuracy * 100.0,
        'mean_accuracy': mean_accuracy,
        'mean_accuracy_percent': mean_accuracy * 100.0,
        'mIoU': mean_iou,
        'mIoU_percent': mean_iou * 100.0,
        'mean_dice': mean_dice,
        'mean_dice_percent': mean_dice * 100.0,
        'foreground_mIoU': foreground_mean_iou,
        'foreground_mIoU_percent': foreground_mean_iou * 100.0,
        'foreground_mean_dice': foreground_mean_dice,
        'foreground_mean_dice_percent': foreground_mean_dice * 100.0,
    }
    for idx, name in enumerate(class_names):
        metrics[f'iou_{name}'] = None if np.isnan(iou[idx]) else float(iou[idx])
        metrics[f'dice_{name}'] = None if np.isnan(dice[idx]) else float(dice[idx])
    return metrics

In [ ]:

summary_dir = Path(OUTPUT_BASE_DIR) / f'{TASK}_{EXPERIMENT_NAME}_multiseed'
summary_dir.mkdir(parents=True, exist_ok=True)
results = []

if TASK == 'semantic':
    LOCKED_METRICS = {
        'mIoU': ('mIoU', 100.0),
        'mDice': ('mean_dice', 100.0),
        'FG-mIoU': ('foreground_mIoU', 100.0),
        'FG-mDice': ('foreground_mean_dice', 100.0),
    }
elif TASK == 'instance':
    LOCKED_METRICS = {
        'AP': ('AP', 1.0),
        'AP50': ('AP50', 1.0),
        'AP75': ('AP75', 1.0),
        'Recall@0.5': ('instance_recall_iou50', 100.0),
        'Dice@0.5': ('mean_instance_dice_iou50', 100.0),
    }
else:
    raise ValueError(f'Unsupported TASK={TASK}')


def cleanup_heavy_artifacts(output_dir):
    output_dir = Path(output_dir)
    patterns = [
        'model_final.pth',
        'model_*.pth',
        'events.out.tfevents*',
        'inference/*.pth',
    ]
    removed = []
    for pattern in patterns:
        for path in output_dir.glob(pattern):
            if path.is_file():
                path.unlink()
                removed.append(str(path))
    if removed:
        print(f'Removed {len(removed)} heavy artifacts from {output_dir}')
    else:
        print(f'No heavy artifacts found in {output_dir}')


def write_multiseed_files(rows, summary_dir):
    summary_dir = Path(summary_dir)
    summary_dir.mkdir(parents=True, exist_ok=True)

    results_json = summary_dir / 'multiseed_results.json'
    results_csv = summary_dir / 'multiseed_results.csv'
    with open(results_json, 'w', encoding='utf-8') as f:
        json.dump(rows, f, indent=2)
    fieldnames = sorted({key for row in rows for key in row.keys()}) if rows else []
    with open(results_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if fieldnames:
            writer.writeheader()
            for row in rows:
                writer.writerow(row)

    summary_rows = []
    ok_rows = [row for row in rows if row.get('status') == 'ok']
    for backbone in BACKBONES:
        backbone_rows = [row for row in ok_rows if row.get('backbone') == backbone]
        for metric_name, (metric_key, scale) in LOCKED_METRICS.items():
            values = [float(row[metric_key]) for row in backbone_rows if row.get(metric_key) not in (None, '')]
            n = len(values)
            if n == 0:
                continue
            mean = sum(values) / n
            std = (sum((value - mean) ** 2 for value in values) / (n - 1)) ** 0.5 if n > 1 else 0.0
            report_mean = mean * scale
            report_std = std * scale
            summary_rows.append({
                'task': TASK,
                'experiment_name': EXPERIMENT_NAME,
                'source_tag': SOURCE_TAG,
                'dataset_tag': DATASET_TAG,
                'backbone': backbone,
                'metric': metric_name,
                'metric_key': metric_key,
                'n': n,
                'mean': mean,
                'std': std,
                'report_mean': report_mean,
                'report_std': report_std,
                'formatted': f'{report_mean:.2f} ± {report_std:.2f}',
            })

    summary_json = summary_dir / 'multiseed_summary.json'
    summary_csv = summary_dir / 'multiseed_summary.csv'
    with open(summary_json, 'w', encoding='utf-8') as f:
        json.dump(summary_rows, f, indent=2)
    summary_fields = sorted({key for row in summary_rows for key in row.keys()}) if summary_rows else []
    with open(summary_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=summary_fields)
        if summary_fields:
            writer.writeheader()
            for row in summary_rows:
                writer.writerow(row)

    print(f'Wrote {results_json}')
    print(f'Wrote {results_csv}')
    print(f'Wrote {summary_json}')
    print(f'Wrote {summary_csv}')
    return summary_rows


for seed in SEEDS:
    SEED = seed
    for backbone in BACKBONES:
        run_row = {
            'task': TASK,
            'experiment_name': EXPERIMENT_NAME,
            'source_tag': SOURCE_TAG,
            'dataset_tag': DATASET_TAG,
            'backbone': backbone,
            'seed': seed,
            'status': 'started',
        }
        output_dir = summary_dir / backbone / f'seed_{seed}'
        try:
            if backbone not in BACKBONE_REGISTRY:
                raise KeyError(f'Unknown backbone {backbone}. Available: {sorted(BACKBONE_REGISTRY)}')
            spec = BACKBONE_REGISTRY[backbone]
            config_file = require_file(path_from_root(CONFIG_ROOT, spec['config']), f'{backbone} config')
            pretrained_weights = require_file(path_from_root(WEIGHTS_ROOT, spec['weights']), f'{backbone} initial weights')
            output_dir.mkdir(parents=True, exist_ok=True)

            train_opts = build_common_opts(backbone, spec, output_dir, TRAIN_EVAL_SPLIT, pretrained_weights)
            print(f'Final override opts for {backbone}, seed={seed}')
            for k, v in zip(train_opts[0::2], train_opts[1::2]):
                print(f'  {k}: {v}')
            train_cmd = [sys.executable, 'train.py', '--num-gpus', str(NUM_GPUS), '--num-machines', str(NUM_MACHINES), '--machine-rank', str(MACHINE_RANK), '--dist-url', DIST_URL, '--config-file', str(config_file)]
            if RESUME:
                train_cmd.append('--resume')
            train_cmd.extend(train_opts)

            if RUN_TRAINING:
                run_streaming_command(train_cmd, WORKING_REPO, output_dir / 'train_command.log')
            else:
                print(f'Skipping training for {backbone}, seed={seed}.')

            checkpoint_override = CHECKPOINTS_BY_BACKBONE.get(backbone)
            eval_weights = Path(checkpoint_override) if checkpoint_override else locate_checkpoint(output_dir, pretrained_weights)
            eval_opts = build_common_opts(backbone, spec, output_dir, EVAL_SPLIT, eval_weights)
            eval_cmd = [sys.executable, 'train.py', '--eval-only', '--num-gpus', str(NUM_GPUS), '--num-machines', str(NUM_MACHINES), '--machine-rank', str(MACHINE_RANK), '--dist-url', DIST_URL, '--config-file', str(config_file)]
            eval_cmd.extend(eval_opts)

            if RUN_EVALUATION:
                run_streaming_command(eval_cmd, WORKING_REPO, output_dir / 'eval_command.log')
                if TASK == 'semantic':
                    pred_json = output_dir / 'inference' / 'sem_seg_predictions.json'
                    metrics = compute_semantic_metrics(pred_json, VERSE_ROOT, EVAL_SPLIT, NUM_CLASSES, CLASS_NAMES)
                    with open(output_dir / 'semantic_metrics.json', 'w', encoding='utf-8') as f:
                        json.dump(metrics, f, indent=2)
                    print('Semantic metrics:', json.dumps(metrics, indent=2))
                else:
                    pred_json = output_dir / 'inference' / 'coco_instances_results.json'
                    metrics = compute_instance_metrics_subprocess(
                        VERSE_ROOT,
                        EVAL_SPLIT,
                        pred_json,
                        output_dir,
                        INSTANCE_MATCH_IOU_THRESHOLD,
                        INSTANCE_SCORE_THRESHOLD,
                    )
                    print('Instance metrics:', json.dumps(metrics, indent=2))
            else:
                metrics = {}
                print(f'Skipping evaluation for {backbone}, seed={seed}.')

            run_row.update({
                'status': 'ok',
                'config_file': str(config_file),
                'initial_weights': str(pretrained_weights),
                'eval_weights': str(eval_weights),
                'output_dir': str(output_dir),
                'eval_split': EVAL_SPLIT,
            })
            run_row.update(metrics)
        except Exception as exc:
            run_row['status'] = 'failed'
            run_row['error'] = repr(exc)
            run_row['output_dir'] = str(output_dir)
            print(f'Backbone {backbone}, seed={seed} failed: {exc}')
            if FAIL_NOTEBOOK_ON_ERROR:
                results.append(run_row)
                write_multiseed_files(results, summary_dir)
                raise
            if STOP_ON_BACKBONE_FAILURE and not SAVE_VERSION_SAFE_MODE:
                results.append(run_row)
                write_multiseed_files(results, summary_dir)
                raise
            print('Continuing so Kaggle Save Version can preserve logs and summary files.')
        finally:
            if output_dir.exists():
                cleanup_heavy_artifacts(output_dir)

        results.append(run_row)
        write_multiseed_files(results, summary_dir)

print('\nFinal per-seed summary:')
print_summary(results)
summary_rows = write_multiseed_files(results, summary_dir)
print('\nMean ± std summary:')
print_summary(summary_rows)


## 6. Cleanup Working Source

In [ ]:
# Remove copied source code and temporary unzipped source after all training/evaluation finishes.
# Experiment outputs under OUTPUT_BASE_DIR are kept.
cleanup_paths = []
if Path(WORKING_REPO).exists():
    cleanup_paths.append(Path(WORKING_REPO))
cleanup_paths.extend(sorted(Path('/kaggle/working').glob('*_unzipped_source')))

for path in cleanup_paths:
    if path.is_dir():
        shutil.rmtree(path)
        print(f'Removed {path}')
    elif path.exists():
        path.unlink()
        print(f'Removed {path}')

print(f'Kept experiment outputs in {OUTPUT_BASE_DIR}')
